In [65]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.svm import SVR
from sklearn.ensemble import AdaBoostRegressor, ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor


from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [66]:
df = pd.read_csv('my_gurgaon_properties_post_feature_selection_v2.csv')

In [67]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Lower
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Medium
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,Higher
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Medium
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Medium


In [68]:
df['furnishing_type'] = df['furnishing_type'].replace({0.0:"unfurnished",1.0:"semifurnished",2.0:"furnished"})
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Lower
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Medium
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,Higher
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Medium
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Medium


In [69]:
X = df.drop(columns=['price'])
y = df['price']

In [70]:
# Applying the log1p transformation to the target variable -> To make the distribution more normal bcos it is right skewed.
y_transformed = np.log1p(y)

## Ordinal Encoding

In [71]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode)
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [72]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(scores.mean(),scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
print(mean_absolute_error(np.expm1(y_test), y_pred))

0.7362632866007919 0.03247179167821135
0.9463324231715656


In [73]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
    ])

    # K-Fold Cross Validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())

    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    output.append(mae)

    return output

In [74]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'Extra Trees Regressor':ExtraTreesRegressor(),
    'Gradient Boosting':GradientBoostingRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'MLP':MLPRegressor(),
    "XgBoost":XGBRegressor()
}

In [75]:
model_output = []
for name, model in model_dict.items():
    model_output.append(scorer(name,model))

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
10,XgBoost,0.895622,0.524353
5,Random Forest,0.881123,0.533558
6,Extra Trees Regressor,0.865881,0.561736
7,Gradient Boosting,0.873445,0.574098
4,Decision Tree,0.772640,0.689333
9,MLP,0.806109,0.775227
8,AdaBoost,0.750696,0.824093
1,svr,0.763946,0.850015
2,Ridge,0.736266,0.946291
0,linear_reg,0.736263,0.946332


## One Hot Encoding

In [76]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat1', OrdinalEncoder(), columns_to_encode),
        ('cat2', OneHotEncoder(drop='first'), ['sector','agePossession', 'furnishing_type'])
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [77]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(scores.mean(),scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
print(mean_absolute_error(np.expm1(y_test), y_pred))

0.8545569356442749 0.01611693171756894
0.6493215723753918


In [78]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
    ])

    # K-Fold Cross Validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())

    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    output.append(mae)

    return output

In [79]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'Extra Trees Regressor':ExtraTreesRegressor(),
    'Gradient Boosting':GradientBoostingRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'MLP':MLPRegressor(),
    "XgBoost":XGBRegressor()
}

In [80]:
model_output = []
for name, model in model_dict.items():
    model_output.append(scorer(name,model))

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
6,Extra Trees Regressor,0.894209,0.474514
10,XgBoost,0.894917,0.485758
5,Random Forest,0.889607,0.495290
7,Gradient Boosting,0.875809,0.566519
9,MLP,0.871240,0.569565
0,linear_reg,0.854557,0.649322
2,Ridge,0.854622,0.652650
4,Decision Tree,0.805545,0.705159
8,AdaBoost,0.751690,0.814178
1,svr,0.769535,0.834948


## Dimensionality Reduction with PCA

In [81]:
columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat1', OrdinalEncoder(), columns_to_encode),
        ('cat2', OneHotEncoder(drop='first', sparse_output=False), ['sector','agePossession','furnishing_type'])
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [82]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(scores.mean(),scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
print(mean_absolute_error(np.expm1(y_test), y_pred))

0.06225318911355489 0.019860463841280027
1.5267072007446847


In [83]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', model)
    ])

    # K-Fold Cross Validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())

    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    output.append(mae)

    return output

In [84]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'Extra Trees Regressor':ExtraTreesRegressor(),
    'Gradient Boosting':GradientBoostingRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'MLP':MLPRegressor(),
    "XgBoost":XGBRegressor()
}

In [85]:
model_output = []
for name, model in model_dict.items():
    model_output.append(scorer(name,model))

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
5,Random Forest,0.759235,0.654400
6,Extra Trees Regressor,0.735708,0.698004
4,Decision Tree,0.692678,0.759610
10,XgBoost,0.622692,0.969020
7,Gradient Boosting,0.609416,0.991495
8,AdaBoost,0.306162,1.360739
1,svr,0.218061,1.361160
9,MLP,0.208483,1.397750
2,Ridge,0.062253,1.526707
0,linear_reg,0.062253,1.526707


## Target Encoding

In [ ]:
import category_encoders as ce

columns_to_encode = ['property_type', 'sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat1', OrdinalEncoder(), columns_to_encode),
        ('cat2', OneHotEncoder(drop='first', sparse_output=False), ['agePossession', 'furnishing_type']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [87]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
print(scores.mean(),scores.std())

X_train, X_test, y_train, y_test = train_test_split(X, y_transformed, test_size=0.2, random_state=42)
pipeline.fit(X_train,y_train)
y_pred = pipeline.predict(X_test)
y_pred = np.expm1(y_pred)
print(mean_absolute_error(np.expm1(y_test), y_pred))

0.8294251800547665 0.01848755039120556
0.7104264338451955


In [88]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', model)
    ])

    # K-Fold Cross Validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')
    
    output.append(scores.mean())

    pipeline.fit(X_train,y_train)
    y_pred = pipeline.predict(X_test)
    y_pred = np.expm1(y_pred)
    mae = mean_absolute_error(np.expm1(y_test), y_pred)

    output.append(mae)

    return output

In [89]:

model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'Ridge':Ridge(),
    'Lasso':Lasso(),
    'Decision Tree':DecisionTreeRegressor(),
    'Random Forest':RandomForestRegressor(),
    'Extra Trees Regressor':ExtraTreesRegressor(),
    'Gradient Boosting':GradientBoostingRegressor(),
    'AdaBoost':AdaBoostRegressor(),
    'MLP':MLPRegressor(),
    "XgBoost":XGBRegressor()
}

In [90]:
model_output = []
for name, model in model_dict.items():
    model_output.append(scorer(name,model))

model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(by='mae')

,name,r2,mae
6,Extra Trees Regressor,0.901247,0.458799
5,Random Forest,0.901008,0.459623
10,XgBoost,0.902470,0.469259
7,Gradient Boosting,0.888966,0.512587
4,Decision Tree,0.826542,0.565597
9,MLP,0.848257,0.611008
8,AdaBoost,0.815344,0.694258
0,linear_reg,0.829425,0.710426
2,Ridge,0.829440,0.710976
1,svr,0.782439,0.818842


In [92]:
# Model to Choose:
# 1. Ordinal Encoding : (Model ---> R2 ---> MAE)
# 	XgBoost ---> 0.895622 ---> 0.524353
# 	Random Forest ---> 0.881123 ---> 0.533558
# 	Extra Trees Regressor ---> 0.865881 ---> 0.561736

# 2. One Hot Encoding : (Model ---> R2 ---> MAE)
# 	Extra Trees Regressor ---> 0.894209 ---> 0.474514
# 	XgBoost ---> 0.894917 ---> 0.485758
# 	Random Forest ---> 0.889607 ---> 0.495290

# 3. One Hot Encoding with PCA : (Model ---> R2 ---> MAE)
# 	Random Forest ---> 0.759235 ---> 0.654400
# 	Extra Trees Regressor ---> 0.735708 ---> 0.698004
# 	Decision Tree ---> 0.692678 ---> 0.759610

# 4. Target Encoding : (Model ---> R2 ---> MAE)
# 	Extra Trees Regressor ---> 0.901247 ---> 0.458799
# 	Random Forest ---> 0.901008 ---> 0.459623
# 	XgBoost ---> 0.902470 ---> 0.469259